# Ensemble Learning
**Chapter 7 — Hands-On ML**

Covers: Voting Classifiers (hard/soft), Bagging, Pasting, OOB Evaluation.

Dataset: `make_moons` — 500 samples, 0.30 noise.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import warnings
warnings.filterwarnings('ignore')

## 1. Dataset

In [ ]:
X, y = make_moons(n_samples=500, noise=0.30, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## 2. Voting Classifier

**Hard voting** — majority class wins.
**Soft voting** — averages predicted probabilities; requires `predict_proba` from all estimators.

In [ ]:
log_clf = LogisticRegression(random_state=42)
rnd_clf = RandomForestClassifier(random_state=42)
svm_clf = SVC(probability=True, random_state=42)

voting_clf = VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='soft',
)
voting_clf.fit(X_train, y_train)

In [ ]:
print(f'{'Classifier':<25} {'Accuracy':>8}')
print('-' * 35)
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    print(f'{clf.__class__.__name__:<25} {accuracy_score(y_test, y_pred):>8.4f}')

## 3. Bagging

Each estimator trains on a random *bootstrap* sample (with replacement). Reduces variance.

In [ ]:
bag_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=100,
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)
bag_clf.fit(X_train, y_train)
acc = accuracy_score(y_test, bag_clf.predict(X_test))
print(f'Bagging accuracy: {acc:.4f}')

## 4. Pasting

Same as bagging but samples are drawn *without* replacement. Less diversity between estimators.

In [ ]:
paste_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=100,
    bootstrap=False,
    n_jobs=-1,
    random_state=42,
)
paste_clf.fit(X_train, y_train)
acc = accuracy_score(y_test, paste_clf.predict(X_test))
print(f'Pasting accuracy: {acc:.4f}')

## 5. Out-of-Bag (OOB) Evaluation

In bagging, ~37% of training samples are never seen by each estimator. These form the OOB set — a free validation without a separate split.

In [ ]:
oob_clf = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=500,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
)
oob_clf.fit(X_train, y_train)
print(f'OOB Score: {oob_clf.oob_score_:.4f}')
acc = accuracy_score(y_test, oob_clf.predict(X_test))
print(f'Test accuracy: {acc:.4f}')